# Lab 04.2 — Ozone tramite API S3

## Introduzione
L'**S3 Gateway** espone i bucket del volume speciale **`/s3v`** con l'API Amazon S3.
Lo stesso dato è quindi accessibile in due modi:

* da Spark/Hadoop con `ofs://om/s3v/<bucket>/...`
* da qualunque client S3 (boto3, AWS CLI, PyArrow, ...) con `s3://<bucket>/...` sull'endpoint `http://s3g:9878`

In questo laboratorio la sicurezza è disattivata: qualunque access key è accettata.

Nota: i bucket creati via S3 hanno di default layout `OBJECT_STORE` (puro object store, non leggibile con `ofs://`).
Il laboratorio imposta `ozone.s3g.default.bucket.layout=FILE_SYSTEM_OPTIMIZED` (vedi `ozone.env`) per usarli da entrambe le parti.

### Obiettivi
1. Creare un `client` boto3 verso l'S3 Gateway e una `SparkSession` verso Ozone.
2. Creare un bucket e caricare oggetti con boto3.
3. Leggere con Spark, via `ofs://`, un oggetto caricato via S3.
4. Scrivere Parquet con Spark e leggerlo con client S3 (boto3, PyArrow).
5. Scaricare un oggetto con boto3 e ispezionarne i metadati.

## Step 1: Client S3 e SparkSession

Creiamo due punti di accesso allo stesso storage:
- un client **boto3** puntato all'endpoint dell'S3 Gateway (`S3_ENDPOINT`, impostato nel `docker-compose.yml`); le credenziali sono fittizie perché la sicurezza è disattivata;
- una **SparkSession** configurata per `ofs://`, come nel notebook precedente.

In [ ]:
import os, boto3
from pyspark.sql import SparkSession, functions as F

# Client S3 verso l'S3 Gateway di Ozone (credenziali fittizie: sicurezza disattivata)
S3_ENDPOINT = os.environ.get("S3_ENDPOINT", "http://s3g:9878")
s3 = boto3.client("s3", endpoint_url=S3_ENDPOINT, region_name="us-east-1",
                  aws_access_key_id="lab", aws_secret_access_key="lab")

# SparkSession con filesystem ofs:// (stessa configurazione del notebook 01)
spark = (SparkSession.builder
         .appName("Lab04-Ozone-S3")
         .master(os.environ.get("SPARK_MASTER", "local[*]"))
         .config("spark.hadoop.fs.ofs.impl", "org.apache.hadoop.fs.ozone.RootedOzoneFileSystem")
         .config("spark.hadoop.ozone.om.address", "om:9862")
         .config("spark.hadoop.ozone.replication", "1")
         .getOrCreate())

## Step 2: Creare un bucket e caricare oggetti con boto3

Creiamo il bucket `dati-s3` (solo se non esiste, così la cella è rieseguibile) e carichiamo il testo di *Alice* con la chiave `testi/alice.txt`.
In S3 non esistono vere directory: `testi/` è solo un prefisso della chiave.
Elenchiamo poi bucket e oggetti con la normale API S3.

In [ ]:
BUCKET = "dati-s3"
# Crea il bucket solo se non esiste già (cella rieseguibile)
if BUCKET not in [b["Name"] for b in s3.list_buckets()["Buckets"]]:
    s3.create_bucket(Bucket=BUCKET)
# Carica il file locale come oggetto "testi/alice.txt"
s3.upload_file("/home/jovyan/datasets/alice_in_wonderland.txt", BUCKET, "testi/alice.txt")

print("Bucket S3:", [b["Name"] for b in s3.list_buckets()["Buckets"]])
for o in s3.list_objects_v2(Bucket=BUCKET)["Contents"]:
    print(f"  {o['Key']:25s} {o['Size']:>10,d} byte")

### Punti chiave
- Un bucket S3 creato via S3 Gateway vive nel volume `/s3v` di Ozone.
- Grazie al layout `FILE_SYSTEM_OPTIMIZED` lo stesso bucket è visibile anche come filesystem `ofs://`.

## Step 3: Spark legge lo stesso oggetto da `ofs://om/s3v/...`

L'oggetto caricato con boto3 è ora letto da Spark senza alcuna copia: cambia solo il percorso, `ofs://om/s3v/<bucket>/<chiave>`.
Contiamo le righe e poi calcoliamo le 5 parole più frequenti (di più di 5 lettere) con l'API DataFrame.

In [ ]:
# Lettura via ofs:// dell'oggetto creato con boto3
testo = spark.read.text(f"ofs://om/s3v/{BUCKET}/testi/alice.txt")
print("Righe lette da Spark:", testo.count())

# Word Count con DataFrame: split su tutto ciò che non è una lettera, poi explode
(testo.select(F.explode(F.split(F.lower("value"), "[^a-z]+")).alias("parola"))
      .where(F.length("parola") > 5)
      .groupBy("parola").count().orderBy(F.desc("count")).show(5))

## Step 4: Spark scrive Parquet, i client S3 lo leggono

Verso opposto: Spark genera un anno di temperature orarie (24 × 365 righe, tre città) e le scrive in Parquet in `ofs://om/s3v/dati-s3/meteo`.
Con boto3 vediamo che ogni file Parquet scritto da Spark è un normale **oggetto S3** (più i file di servizio come `_SUCCESS`).

In [ ]:
# Un anno di temperature orarie per tre città (dati sintetici, seed fissi)
temperature = (spark.range(0, 24 * 365)
               .withColumn("ora", F.expr("timestamp_seconds(1704067200 + id * 3600)"))
               .withColumn("citta", F.element_at(F.array(F.lit("Roma"), F.lit("Milano"), F.lit("Napoli")),
                                                 (F.col("id") % 3 + 1).cast("int")))
               .withColumn("gradi", F.round(15 + 10 * F.sin(F.col("id") / 600) + F.randn(3) * 2, 1))
               .drop("id"))

# Scrittura via ofs://: i file risultanti sono oggetti S3 nello stesso bucket
temperature.write.mode("overwrite").parquet(f"ofs://om/s3v/{BUCKET}/meteo")

# Gli stessi file visti dall'API S3
for o in s3.list_objects_v2(Bucket=BUCKET, Prefix="meteo/")["Contents"]:
    print(f"  {o['Key']:75s} {o['Size']:>8,d} byte")

### Step 4b: Leggere il Parquet con PyArrow via S3

**PyArrow** è un client S3 indipendente da Spark e da Hadoop: legge la cartella Parquet direttamente dall'endpoint dell'S3 Gateway.
Converte il risultato in pandas e calcola minimo, media e massimo delle temperature per città.

In [ ]:
# PyArrow legge la cartella Parquet direttamente via S3 (senza Spark)
import pyarrow.fs as pafs, pyarrow.parquet as pq

s3fs = pafs.S3FileSystem(endpoint_override=S3_ENDPOINT, scheme="http", region="us-east-1",
                         access_key="lab", secret_key="lab")
tabella = pq.read_table(f"{BUCKET}/meteo", filesystem=s3fs)
df = tabella.to_pandas()
print(tabella.num_rows, "righe lette con PyArrow via S3")
# Statistiche per città (l'ultima espressione della cella viene mostrata come output)
df.groupby("citta")["gradi"].agg(["min", "mean", "max"]).round(1)

### Punti chiave
- Lo stesso dato è accessibile da **Spark** (`ofs://`) e da **qualunque client S3** (boto3, PyArrow, AWS CLI) senza copie.
- Ozone consente di condividere dati tra applicazioni Hadoop e strumenti nativi S3.

## Step 5: Scaricare un oggetto con boto3

Scarichiamo in locale uno dei file `part-*` del Parquet scritto da Spark e ne leggiamo i metadati con `pq.read_metadata` (righe, gruppi di righe, schema).
È la prova che il file è un normale Parquet, non un formato proprietario di Ozone.

In [ ]:
# Primo file "part-*" del Parquet scritto da Spark
nome = [o["Key"] for o in s3.list_objects_v2(Bucket=BUCKET, Prefix="meteo/part-")["Contents"]][0]
s3.download_file(BUCKET, nome, "/tmp/meteo.parquet")
# Metadati del file Parquet scaricato
print(pq.read_metadata("/tmp/meteo.parquet"))

### Punti chiave
- `download_file` restituisce l'oggetto così com'è: un file Parquet standard, leggibile anche fuori da Spark.

## Step 6: Chiudere la SparkSession

Liberiamo le risorse del driver. I dati restano in Ozone finché non si eseguono `docker compose down -v`.

In [ ]:
spark.stop()

## Riepilogo

| Concetto | API o comando | A cosa serve |
|---|---|---|
| S3 Gateway | endpoint `http://s3g:9878`, volume `/s3v` | esporre i bucket Ozone con l'API S3 |
| Client S3 | `boto3.client("s3", endpoint_url=...)` | creare bucket, caricare e scaricare oggetti |
| Spark su bucket S3 | `ofs://om/s3v/<bucket>/<chiave>` | leggere/scrivere con Spark gli stessi dati |
| Layout bucket | `ozone.s3g.default.bucket.layout=FILE_SYSTEM_OPTIMIZED` | rendere i bucket S3 accessibili anche con `ofs://` |
| PyArrow | `pafs.S3FileSystem` + `pq.read_table` | leggere Parquet via S3 senza Spark |
| Download oggetto | `s3.download_file`, `pq.read_metadata` | ispezionare un file Parquet scritto da Spark |